# Fact Table

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

In [0]:
df = spark.read.format("delta").load("abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")
display(df)

In [0]:
df_src = df.drop("BranchName","DealerName","Product_Name","ModelType","Date")

# Read dimension tables

In [0]:
dim_model = spark.read.table("carsalescatalog.gold.dim_model")
dim_dealer = spark.read.table("carsalescatalog.gold.dim_dealer")
dim_branch = spark.read.table("carsalescatalog.gold.dim_branch")
dim_date = spark.read.table("carsalescatalog.gold.dim_date")

In [0]:
print(dim_model.count())
print(dim_branch.count())
print(dim_dealer.count())
print(dim_date.count())

In [0]:
df_fact = df_src.join(dim_branch, df_src["Branch_ID"]==dim_branch["Branch_ID"],"left")\
      .join(dim_dealer, df_src["Dealer_ID"]==dim_dealer["Dealer_ID"],"left")\
      .join(dim_model, df_src["Model_ID"]==dim_model["Model_ID"], "left")\
      .join(dim_date, df_src["Date_ID"]==dim_date["Date_ID"],"left")\
      .select(dim_branch.dim_branch_key, dim_dealer.dim_dealer_key, dim_model.dim_model_key, dim_date.dim_date_key, df_src.Revenue, df_src.Units_Sold, df_src.RevPerUnit)

In [0]:
if DeltaTable.isDeltaTable(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/factsales/"):
  deltatable = DeltaTable.forPath(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/factsales/")

  deltatable.alias("trg").merge(df_fact.alias("src"),\
      """trg.dim_branch_key = src.dim_branch_key and
         trg.dim_dealer_key = src.dim_dealer_key and
         trg.dim_model_key = src.dim_model_key and
         trg.dim_date_key = src.dim_date_key""")\
             .whenMatchedUpdateAll()\
             .whenNotMatchedInsertAll()\
                 .execute()
else:
  df_fact.write.mode("overwrite").format("delta").save("abfss://gold@adlscarsales.dfs.core.windows.net/factsales/")

  spark.sql("""
            create table carsalescatalog.gold.factsales
            using delta
            location 'abfss://gold@adlscarsales.dfs.core.windows.net/factsales/'
            """)

In [0]:
spark.sql("select * from carsalescatalog.gold.factsales").display()